In [1]:
#!/usr/bin/env python3
"""
=============================================================================
Algorithm 1: Quantum Parameter Estimation for Noise Control
=============================================================================
Paper: "Stochastic Differential Equations Approaches to
        Quantum System Noise Analysis and Control"

Physics model (eq. 2):
    dU(t) = [ -(iH + e2V2/2) dt  -  ie V dB(t) ] U(t),   U(0) = I

where H is the system Hamiltonian, V = sum_k theta_k V_k is the noise
potential, B(t) is standard Brownian motion, e = epsilon is perturbation.

=== NOTE ON SYMMETRY ===
For H = (w/2)sigma_z and V = theta_1*sigma_x + theta_2*sigma_y (paper),
there is a U(1) rotational symmetry: H generates exactly the rotation that
mixes sigma_x and sigma_y, so V^2 = (theta_1^2+theta_2^2)*I regardless of
the mixing angle. All O(eps^2) corrections then depend ONLY on |theta|^2,
making it impossible to identify theta_1 and theta_2 separately.

Fix: add a small transverse field  g*sigma_x  to H. This breaks the XY
symmetry so sigma_x and sigma_y perturbations have different effects.
=============================================================================

Key steps
---------
  1. Expand U(t) = U0 + e*U1 + e^2*U2 + O(e^3)    (eqs 7-10)
  2. U0(t) = exp(-itH)                              (eq 11)
  3. E[U1(t)] = 0  (Ito martingale property)        (eq 14)
  4. E[U*(t)XU(t)] up to O(e^2) via quadrature      (eq 18)
  5. Analytical doubly-time-averaged Q matrix        (eqs 23-24)
  6. BFGS optimisation over multi-observable loss    -> recover theta
  7. Euler-Maruyama SDE simulation                   (independent check)
=============================================================================
"""

import numpy as np
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import time

# ============================================================
# SECTION 1 -- Operators and system parameters
# ============================================================

# Pauli matrices
sx = np.array([[0, 1],   [1,  0]],  dtype=complex)
sy = np.array([[0, -1j], [1j, 0]],  dtype=complex)
sz = np.array([[1, 0],   [0, -1]],  dtype=complex)
I2 = np.eye(2, dtype=complex)

# System Hamiltonian
# Paper: H = (w/2)*sigma_z  ->  U(1) symmetry, theta_k unidentifiable.
# Fix:   H = (w/2)*sigma_z + g*sigma_x  (breaks the symmetry).
omega = 1.0
g     = 0.20          # transverse field that breaks the XY symmetry
H     = 0.5 * omega * sz + g * sx

# Noise operator basis and true parameters
V_ops      = [sx, sy]
theta_true = np.array([0.25, 0.15])   # parameters to recover
epsilon    = 0.15                      # perturbation strength

# Initial state and observables
rho0     = np.array([[0, 0], [0, 1]], dtype=complex)   # ground state |1><1|
obs_list = [sz, sx, sy]

# ============================================================
# SECTION 2 -- Eigenprojections  P_a = |a><a|  of H
# ============================================================

eigvals, eigvecs = np.linalg.eigh(H)      # ascending order
n_eig = len(eigvals)
P = [eigvecs[:, i:i+1] @ eigvecs[:, i:i+1].conj().T for i in range(n_eig)]

# Sanity checks
assert np.allclose(sum(P), I2),        "Projections must sum to I"
assert np.allclose(P[0] @ P[1], 0),   "Projections must be orthogonal"
assert np.allclose(P[0] @ P[0], P[0]),"P must be idempotent"
assert np.allclose(sum(eigvals[a] * P[a] for a in range(n_eig)), H), \
    "Spectral decomposition check"


# ============================================================
# SECTION 3 -- Unperturbed evolution  U0(t) = exp(-itH)   (eq 11)
# ============================================================

def U0(t):
    """Return U0(t) = exp(-itH) as a 2x2 complex matrix."""
    return sum(np.exp(-1j * eigvals[a] * t) * P[a] for a in range(n_eig))


def V_total(theta):
    """Return noise potential V = sum_k theta_k V_k."""
    return sum(theta[k] * V_ops[k] for k in range(len(theta)))


# Verify unitarity
for _t in [0.5, 1.23, 7.7]:
    _U = U0(_t)
    assert np.allclose(_U @ _U.conj().T, I2, atol=1e-10), \
        f"U0 not unitary at t={_t}"


# ============================================================
# SECTION 4 -- E[U*(t) X U(t)] via numerical quadrature (eq 18)
#
#   E[U*XU] = U0*(t) X U0(t)
#     + e^2 {
#         -1/2 * int_0^t  U0*(t) X U0(t-tau) V^2 U0(tau) dtau        [A]
#         -1/2 * int_0^t  U0*(tau) V^2 U0*(t-tau) X U0(t) dtau        [B]
#         +     int_0^t  U0*(tau) V U0*(t-tau) X U0(t-tau) V U0(tau) dtau [C]
#       }
#
#   Term C comes from Ito isometry applied to E[U1*(t) X U1(t)]:
#     E[U1*(t)XU1(t)] = int_0^t U0*(tau)V U0*(t-tau) X U0(t-tau) V U0(tau) dtau
# ============================================================

def E_UXU(t, theta, X, n_pts=400):
    """
    Compute E[U*(t) X U(t)] up to O(epsilon^2) using eq 18.
    Numerical integration by trapezoidal rule.
    """
    V   = V_total(theta)
    V2  = V @ V
    U0t = U0(t)
    Ud  = U0t.conj().T                    # U0*(t)

    taus = np.linspace(0.0, t, n_pts)
    h    = taus[1] - taus[0] if n_pts > 1 else t
    w    = np.ones(n_pts);  w[0] = 0.5;  w[-1] = 0.5   # trapezoidal

    IA = np.zeros((2, 2), dtype=complex)
    IB = np.zeros((2, 2), dtype=complex)
    IC = np.zeros((2, 2), dtype=complex)

    for i, tau in enumerate(taus):
        U0m   = U0(t - tau)               # U0(t - tau)
        U0tau = U0(tau)                   # U0(tau)
        U0md  = U0m.conj().T              # U0*(t - tau)
        U0td  = U0tau.conj().T            # U0*(tau)

        IA += w[i] * (Ud @ X @ U0m @ V2 @ U0tau)                       # Term A
        IB += w[i] * (U0td @ V2 @ U0md @ X @ U0t)                      # Term B
        IC += w[i] * (U0td @ V @ U0md @ X @ U0m @ V @ U0tau)           # Term C

    IA *= h;  IB *= h;  IC *= h
    return Ud @ X @ U0t + epsilon**2 * (-0.5 * IA - 0.5 * IB + IC)


# Verify zero-epsilon limit
_eps = epsilon
epsilon = 0.0
for _t in [1.0, 5.0]:
    _E   = E_UXU(_t, theta_true, sz)
    _ref = U0(_t).conj().T @ sz @ U0(_t)
    assert np.allclose(_E, _ref, atol=1e-5), f"Zero-e limit failed at t={_t}"
epsilon = _eps

# Verify Hermiticity
for _t in [1.0, 5.0]:
    for _X in [sz, sx]:
        _E = E_UXU(_t, theta_true, _X)
        assert np.allclose(_E, _E.conj().T, atol=1e-7), "E[U*XU] not Hermitian"


# ============================================================
# SECTION 5 -- Analytical Q matrix  (eqs 23-24)
#
#   After doubly time-averaging to remove oscillatory terms:
#
#   Q = sum_{k,m} theta_k theta_m {
#         sum_a  [ P_a X P_a V_k V_m P_a           (term 1)
#                + P_a V_k V_m P_a X P_a           (term 2)
#                - P_a V_k P_a X P_a V_m P_a ]     (term 3)
#       - sum_{a,b}  P_a V_k P_b X P_b V_m P_a }   (term 4)
# ============================================================

def Q_matrix(theta, X):
    """Return the analytical Q matrix from eq 24."""
    Q = np.zeros((2, 2), dtype=complex)
    for k in range(len(theta)):
        for m in range(len(theta)):
            Vk, Vm = V_ops[k], V_ops[m]
            VkVm   = Vk @ Vm
            blk    = np.zeros((2, 2), dtype=complex)
            for Pa in P:
                blk += Pa @ X @ Pa @ VkVm @ Pa              # term 1
                blk += Pa @ VkVm @ Pa @ X @ Pa              # term 2
                blk -= Pa @ Vk @ Pa @ X @ Pa @ Vm @ Pa      # term 3
            for Pa in P:
                for Pb in P:
                    blk -= Pa @ Vk @ Pb @ X @ Pb @ Vm @ Pa  # term 4
            Q += theta[k] * theta[m] * blk
    return Q


# Verify Q is Hermitian
for _X in [sz, sx, sy]:
    _Q = Q_matrix(theta_true, _X)
    assert np.allclose(_Q, _Q.conj().T, atol=1e-12), "Q must be Hermitian"


# ============================================================
# SECTION 6 -- Measurement vector
#
#   Combines:
#   (a) Time-dependent: Tr(rho0 * E[U*(t_j) X U(t_j)])  -- oscillatory info
#   (b) Time-averaged:  Tr(rho0 * Q_X)                  -- quadratic in theta
# ============================================================

meas_times = np.array([0.5, 1.0, 2.0, 5.0, 10.0, 20.0])


def all_measurements(theta):
    """Full measurement vector: 6 times x 3 observables + 3 Q values = 21."""
    vals = []
    for t in meas_times:
        for X in obs_list:
            M = E_UXU(t, theta, X)
            vals.append(float(np.real(np.trace(rho0 @ M))))
    for X in obs_list:
        Qm = Q_matrix(theta, X)
        vals.append(float(np.real(np.trace(rho0 @ Qm))))
    return np.array(vals)


# ============================================================
# SECTION 7 -- Target measurements and loss function
# ============================================================

print("=" * 68)
print(" Algorithm 1 -- Quantum Parameter Estimation via Ito SDE")
print("=" * 68)
print(f"  H = (w/2)*sz + g*sx,  w={omega},  g={g}")
print(f"  V = theta_1*sx + theta_2*sy")
print(f"  theta_true = {theta_true}")
print(f"  epsilon    = {epsilon}")
print(f"  Initial state: ground state |1><1|")
print()

print("Computing target measurements ...", end=" ", flush=True)
t0      = time.time()
targets = all_measurements(theta_true)
print(f"done ({time.time()-t0:.1f}s, {len(targets)} measurements)")
print(f"  Sample values (first 5): {np.round(targets[:5], 5)}")

# Identifiability check: targets must change for different theta
_d1 = np.abs(all_measurements(theta_true + [0.05, 0.0]) - targets).max()
_d2 = np.abs(all_measurements(theta_true + [0.0, 0.05]) - targets).max()
print(f"  Sensitivity: d(theta_1+0.05)={_d1:.4f},  d(theta_2+0.05)={_d2:.4f}")
assert _d1 > 1e-5 and _d2 > 1e-5, "Not sensitive to both parameters!"
print()

loss_hist  = []
theta_hist = []


def loss(theta):
    meas = all_measurements(theta)
    L    = float(np.dot(meas - targets, meas - targets))
    loss_hist.append(L)
    theta_hist.append(theta.copy())
    return L


# ============================================================
# SECTION 8 -- BFGS Optimisation  (Algorithm 1 outer loop)
# ============================================================

theta_init = np.array([0.40, 0.40])
print(f"Starting BFGS from theta_init = {theta_init}")
L0 = loss(theta_init)
print(f"  Initial loss = {L0:.6e}\n")

# NOTE on sign degeneracy
# ─────────────────────────────────────────────────────────────────────────
# At O(epsilon^2), every term in E[U*XU] contains V an even number of times
# (V^2, V...V from the Ito isometry), so theta and -theta give identical
# measurements. This is an inherent limitation of the second-order perturbative
# framework. We break it with non-negativity bounds — physically justified
# when theta_k are noise amplitudes (power spectral density coefficients >= 0).
# ─────────────────────────────────────────────────────────────────────────
result = minimize(loss, theta_init, method="L-BFGS-B",
                  bounds=[(0, None), (0, None)],
                  options={"ftol": 1e-15, "gtol": 1e-12,
                           "maxiter": 5000, "disp": False})

theta_est = result.x
abs_err   = np.abs(theta_est - theta_true)
rel_err   = 100.0 * abs_err / np.abs(theta_true)

print("=" * 68)
print(" OPTIMISATION RESULTS")
print("=" * 68)
print(f"  True parameters  :  theta_1 = {theta_true[0]:.6f}   theta_2 = {theta_true[1]:.6f}")
print(f"  Initial guess    :  theta_1 = {theta_init[0]:.6f}   theta_2 = {theta_init[1]:.6f}")
print(f"  Estimated params :  theta_1 = {theta_est[0]:.6f}   theta_2 = {theta_est[1]:.6f}")
print(f"  Absolute error   :  d1 = {abs_err[0]:.2e}   d2 = {abs_err[1]:.2e}")
print(f"  Relative error   :  d1 = {rel_err[0]:.3f}%   d2 = {rel_err[1]:.3f}%")
print(f"  Final loss       :  {result.fun:.2e}")
print(f"  Converged        :  {result.success}  ({result.message})")
print(f"  Iterations       :  {result.nit}\n")


# ============================================================
# SECTION 9 -- Euler-Maruyama SDE validation  (eq 2 directly)
#
#   U_{n+1} = U_n + [-(iH + e^2 V^2/2)] U_n dt + (-ie V U_n) dW_n
#   where dW_n ~ N(0, dt)
# ============================================================

print("Running Euler-Maruyama SDE simulation ...", end=" ", flush=True)
t0 = time.time()


def sde_simulate(theta, T=10.0, dt=0.005, n_paths=8000, seed=42):
    rng    = np.random.default_rng(seed)
    V      = V_total(theta)
    drift  = -(1j * H + 0.5 * epsilon**2 * V @ V)
    diffop = -1j * epsilon * V
    nsteps = int(T / dt)
    sq_dt  = np.sqrt(dt)
    acc    = {k: np.zeros((2, 2), dtype=complex) for k in ["sz", "sx", "sy"]}
    for _ in range(n_paths):
        U = I2.copy()
        for __ in range(nsteps):
            dW = rng.standard_normal() * sq_dt
            U += drift @ U * dt + diffop @ U * dW
        Ud = U.conj().T
        acc["sz"] += Ud @ sz @ U
        acc["sx"] += Ud @ sx @ U
        acc["sy"] += Ud @ sy @ U
    return {k: float(np.real(np.trace(rho0 @ acc[k] / n_paths))) for k in acc}


T_val    = 10.0
sde_res  = sde_simulate(theta_true, T=T_val, dt=0.01, n_paths=3000)
pert_res = {
    "sz": float(np.real(np.trace(rho0 @ E_UXU(T_val, theta_true, sz)))),
    "sx": float(np.real(np.trace(rho0 @ E_UXU(T_val, theta_true, sx)))),
    "sy": float(np.real(np.trace(rho0 @ E_UXU(T_val, theta_true, sy)))),
}
print(f"done ({time.time()-t0:.1f}s)\n")

print(f"Validation at T={T_val}  (SDE Monte Carlo vs O(eps^2) perturbation)")
print(f"  {'Observable':<25} {'SDE (MC)':>10}   {'Perturbation':>14}")
print(f"  {'─'*52}")
lbl = {"sz": "Tr(rho0 E[U*sz U])",
       "sx": "Tr(rho0 E[U*sx U])",
       "sy": "Tr(rho0 E[U*sy U])"}
for k in ["sz", "sx", "sy"]:
    diff = abs(sde_res[k] - pert_res[k])
    print(f"  {lbl[k]:<25} {sde_res[k]:+10.6f}   {pert_res[k]:+14.6f}"
          f"   |diff|={diff:.4f}")
print()


# ============================================================
# SECTION 10 -- Figures
# ============================================================

fig = plt.figure(figsize=(16, 10))
fig.suptitle(
    "Algorithm 1: Quantum Parameter Estimation via Ito SDE Perturbation\n"
    "dU = [-(iH + eps2*V2/2) dt - i*eps*V dB] U,  "
    "V = theta1*sigma_x + theta2*sigma_y",
    fontsize=12, fontweight="bold"
)

gs = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Panel 1: Loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.semilogy(loss_hist, color="royalblue", lw=1.5)
ax1.set_xlabel("Iteration"); ax1.set_ylabel("Loss (log)")
ax1.set_title("Convergence History", fontweight="bold")
ax1.grid(True, which="both", alpha=0.3)
ax1.annotate(f"Final: {result.fun:.1e}",
             xy=(len(loss_hist)-1, loss_hist[-1]),
             xytext=(0.4, 0.7), textcoords="axes fraction", fontsize=9,
             arrowprops=dict(arrowstyle="->", lw=0.8))

# Panel 2: theta trace
ax2 = fig.add_subplot(gs[0, 1])
th  = np.array(theta_hist)
ax2.plot(th[:, 0], lw=1.5, color="steelblue",  label="theta_1 (est)")
ax2.plot(th[:, 1], lw=1.5, color="darkorange", label="theta_2 (est)")
ax2.axhline(theta_true[0], ls="--", color="steelblue",  alpha=0.6,
            label=f"theta_1 true={theta_true[0]}")
ax2.axhline(theta_true[1], ls="--", color="darkorange", alpha=0.6,
            label=f"theta_2 true={theta_true[1]}")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Parameter value")
ax2.set_title("Parameter Convergence", fontweight="bold")
ax2.legend(fontsize=8, ncol=2); ax2.grid(True, alpha=0.3)

# Panel 3: Bar chart comparison
ax3 = fig.add_subplot(gs[0, 2])
x_pos = np.arange(2);  bw = 0.30
b1 = ax3.bar(x_pos - bw/2, theta_true, bw, label="True",
             color="steelblue",  edgecolor="k", alpha=0.85)
b2 = ax3.bar(x_pos + bw/2, theta_est,  bw, label="Estimated",
             color="darkorange", edgecolor="k", alpha=0.85)
for b, v in zip(list(b1)+list(b2), list(theta_true)+list(theta_est)):
    ax3.text(b.get_x()+b.get_width()/2, v+0.004, f"{v:.5f}",
             ha="center", fontsize=8)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(["theta_1 (sx)", "theta_2 (sy)"])
ax3.set_ylabel("Parameter value"); ax3.set_ylim(0, 0.40)
ax3.set_title("True vs Estimated Parameters", fontweight="bold")
ax3.legend(); ax3.grid(True, axis="y", alpha=0.3)

# Panel 4: Observable time evolution
ax4 = fig.add_subplot(gs[1, :2])
t_fine = np.linspace(0.2, 30, 60)
for obs, name, col in [(sz, "sigma_z", "royalblue"),
                       (sx, "sigma_x", "tomato")]:
    m_tr = [np.real(np.trace(rho0 @ E_UXU(t, theta_true, obs, n_pts=150)))
            for t in t_fine]
    m_es = [np.real(np.trace(rho0 @ E_UXU(t, theta_est,  obs, n_pts=150)))
            for t in t_fine]
    ax4.plot(t_fine, m_tr, "-",  color=col, lw=2.0, label=f"E[{name}] true")
    ax4.plot(t_fine, m_es, "--", color=col, lw=1.5, alpha=0.7,
             label=f"E[{name}] est.")
for _t in meas_times:
    ax4.axvline(_t, color="gray", lw=0.7, ls=":", alpha=0.4)
ax4.set_xlabel("Time t"); ax4.set_ylabel("Tr(rho0 * E[U*XU])")
ax4.set_title("Observable Evolution (dashed = estimated theta)", fontweight="bold")
ax4.legend(fontsize=9, ncol=2); ax4.grid(True, alpha=0.3)

# Panel 5: SDE vs theory
ax5 = fig.add_subplot(gs[1, 2])
labels = ["sigma_z", "sigma_x", "sigma_y"]
sde_v  = [sde_res["sz"],  sde_res["sx"],  sde_res["sy"]]
pert_v = [pert_res["sz"], pert_res["sx"], pert_res["sy"]]
x5 = np.arange(3);  w5 = 0.30
ax5.bar(x5-w5/2, sde_v,  w5, label="SDE (Monte Carlo)",
        color="mediumseagreen", edgecolor="k", alpha=0.85)
ax5.bar(x5+w5/2, pert_v, w5, label="O(eps^2) theory",
        color="mediumpurple",   edgecolor="k", alpha=0.85)
ax5.set_xticks(x5); ax5.set_xticklabels(labels)
ax5.set_ylabel("Measurement value")
ax5.set_title(f"SDE Validation at T={T_val}", fontweight="bold")
ax5.legend(fontsize=9); ax5.grid(True, axis="y", alpha=0.3)

out_path = "quantum_sde_algorithm.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Figure saved -> {out_path}")

# ============================================================
# SECTION 11 -- Summary and self-checks
# ============================================================

print()
print("=" * 68)
print(" SUMMARY")
print("=" * 68)
print(f"  Measurement count     : {len(targets)}")
print(f"  Sampling times        : {meas_times}")
print(f"  True theta            : {theta_true}")
print(f"  Estimated theta       : {np.round(theta_est, 6)}")
print(f"  Max absolute error    : {abs_err.max():.2e}")
print(f"  Max relative error    : {rel_err.max():.3f}%")
print()

assert abs_err[0] < 1e-2, f"theta_1 error too large: {abs_err[0]:.2e}"
assert abs_err[1] < 1e-2, f"theta_2 error too large: {abs_err[1]:.2e}"
print("  Self-check PASSED: both parameters recovered to < 1% absolute error")
print()
print("Algorithm 1 completed successfully."
      if result.success else
      f"Note: BFGS flag={result.success}, final loss={result.fun:.2e}")

 Algorithm 1 -- Quantum Parameter Estimation via Ito SDE
  H = (w/2)*sz + g*sx,  w=1.0,  g=0.2
  V = theta_1*sx + theta_2*sy
  theta_true = [0.25 0.15]
  epsilon    = 0.15
  Initial state: ground state |1><1|

Computing target measurements ... done (0.2s, 21 measurements)
  Sample values (first 5): [-0.9786  -0.04866  0.19015 -0.92387 -0.18063]
  Sensitivity: d(theta_1+0.05)=0.0613,  d(theta_2+0.05)=0.0453

Starting BFGS from theta_init = [0.4 0.4]
  Initial loss = 4.295568e-01



/var/folders/mc/10937dzd61z0jsl42wgdszzr0000gn/T/ipykernel_23866/1209965950.py:288: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(loss, theta_init, method="L-BFGS-B",


 OPTIMISATION RESULTS
  True parameters  :  theta_1 = 0.250000   theta_2 = 0.150000
  Initial guess    :  theta_1 = 0.400000   theta_2 = 0.400000
  Estimated params :  theta_1 = 0.249998   theta_2 = 0.150004
  Absolute error   :  d1 = 2.48e-06   d2 = 3.57e-06
  Relative error   :  d1 = 0.001%   d2 = 0.002%
  Final loss       :  6.16e-15
  Converged        :  True  (CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH)
  Iterations       :  29

Running Euler-Maruyama SDE simulation ... done (15.5s)

Validation at T=10.0  (SDE Monte Carlo vs O(eps^2) perturbation)
  Observable                  SDE (MC)     Perturbation
  ────────────────────────────────────────────────────
  Tr(rho0 E[U*sz U])         -0.822102        -0.802799   |diff|=0.0193
  Tr(rho0 E[U*sx U])         -0.426255        -0.407242   |diff|=0.0190
  Tr(rho0 E[U*sy U])         -0.369036        -0.355406   |diff|=0.0136

Figure saved -> quantum_sde_algorithm.png

 SUMMARY
  Measurement count     : 21
  Sampling times      